In [1]:
# Install all packages required by this notebook.
%pip install -q matplotlib tensorflow opencv-python scikit-learn

In [2]:
import kagglehub


# Download latest version
path = kagglehub.dataset_download("ajaysonicu/deepfakefusion-399k-realfake-faces")

print("Path to dataset files:", path)

 20%|██        | 1.36G/6.67G [00:36<02:20, 40.5MB/s]


KeyboardInterrupt: 

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Activation, Dropout, BatchNormalization

SEED = 42
# Seeds Python's `random`, NumPy and TensorFlow in one call
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
from pathlib import Path

dataset_root = Path(path) / "deepfakefusion_v2_sixsource_dataset"

print("Dataset root:", dataset_root)

for item in sorted(dataset_root.iterdir()):
    print("[DIR] " if item.is_dir() else "[FILE]", item.name)

for item in sorted(dataset_root.rglob("*")):
    relative_path = item.relative_to(dataset_root)

    if item.is_dir() and len(relative_path.parts) <= 3:
        print("[DIR] " if item.is_dir() else "[FILE]", relative_path)

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

image_files = [
    file for file in dataset_root.rglob("*")
    if file.is_file() and file.suffix.lower() in image_extensions
]

print("Total images:", len(image_files))

for file in image_files[:20]:
    print(file.relative_to(dataset_root))

In [ ]:
IMG_SIZE = (128, 128)
batch_size = 32

train_dir = dataset_root / "train"
val_dir = dataset_root / "val"
test_dir = dataset_root / "test"

# Class names come from the sub-folder names (alphabetical: fake=0, real=1),
# label_mode="binary" gives 0/1 float labels that match Dense(1, sigmoid).
train_full_ds = tf.keras.utils.image_dataset_from_directory(
    str(train_dir),
    image_size=IMG_SIZE,
    batch_size=batch_size,
    label_mode="binary",
    shuffle=True,
    seed=SEED,
)

# Shuffle validation too: with shuffle=False, taking the first N batches
# would only return images from the first class folder ("fake").
val_full_ds = tf.keras.utils.image_dataset_from_directory(
    str(val_dir),
    image_size=IMG_SIZE,
    batch_size=batch_size,
    label_mode="binary",
    shuffle=True,
    seed=SEED,
)

# Test set stays unshuffled so predictions line up with test_ds.file_paths
test_ds = tf.keras.utils.image_dataset_from_directory(
    str(test_dir),
    image_size=IMG_SIZE,
    batch_size=batch_size,
    label_mode="binary",
    shuffle=False,
)

class_names = train_full_ds.class_names
print("Classes:", {name: i for i, name in enumerate(class_names)})
print("Training images:", len(train_full_ds.file_paths))
print("Validation images:", len(val_full_ds.file_paths))
print("Test images:", len(test_ds.file_paths))

In [ ]:
from IPython.display import clear_output

class LiveTrainingPlot(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.train_loss = []
        self.val_loss = []
        self.train_acc = []
        self.val_acc = []

        self.best_val_loss = float('inf')
        self.best_epoch = 0
        self.best_val_acc = 0

    def on_epoch_end(self, epoch, logs=None):

        # Current epoch metrics
        train_loss = logs['loss']
        val_loss = logs['val_loss']
        train_acc = logs['accuracy']
        val_acc = logs['val_accuracy']

        # Save epoch metrics
        self.train_loss.append(logs['loss'])
        self.val_loss.append(logs['val_loss'])
        self.train_acc.append(logs['accuracy'])
        self.val_acc.append(logs['val_accuracy'])

        # Check if this is the best epoch so far
        if val_loss < self.best_val_loss:
            self.best_val_loss = val_loss
            self.best_val_acc = val_acc
            self.best_epoch = epoch + 1

        best_epoch_idx = self.best_epoch - 1

        # Clear previous plot
        clear_output(wait=True)
        epochs_completed = range(1, len(self.train_loss) + 1)
        plt.figure(figsize=(12, 5))

        # Loss curves
        plt.subplot(1, 2, 1)
        plt.plot(
            epochs_completed,
            self.train_loss,
            marker='o',
            label='Training Loss'
        )
        plt.plot(
            epochs_completed,
            self.val_loss,
            marker='o',
            label='Validation Loss'
        )
        plt.xlabel('Epoch')
        plt.ylabel('Binary Cross-Entropy Loss')
        plt.title('Training vs Validation Loss')
        plt.legend()
        plt.grid(True)
        # Accuracy curves
        plt.subplot(1, 2, 2)
        plt.plot(
            epochs_completed,
            self.train_acc,
            marker='o',
            label='Training Accuracy'
        )
        plt.plot(
            epochs_completed,
            self.val_acc,
            marker='o',
            label='Validation Accuracy'

        )
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.title('Training vs Validation Accuracy')
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()



        print(f"Previous EPOCH: {epoch + 1}")
        print(f"Train Loss: {train_loss:.4f} | "f"Train Accuracy: {train_acc:.4f}")
        print(f"Val Loss:   {val_loss:.4f} | "f"Val Accuracy:   {val_acc:.4f}")

        print(f"\nBEST EPOCH SO FAR: {self.best_epoch}")
        print(f"Train Loss: {self.train_loss[best_epoch_idx]:.4f} | "f"Train Accuracy: {self.train_acc[best_epoch_idx]:.4f}")
        print(f"Val Loss: {self.best_val_loss:.4f} | "f"Val Accuracy: {self.best_val_acc:.4f}")

    def on_train_end(self, logs=None):
        print(f"Training completed for {len(self.train_loss)} epochs")
        print(f"Best Epoch: {self.best_epoch}\n" f"Best Validation Loss: {self.best_val_loss:.4f}\n" f"Validation Accuracy at Best Epoch: {self.best_val_acc:.4f}")

        print("\nEarlyStopping restored the model weights " f"from Epoch {self.best_epoch}.")

In [ ]:
# Sampling the training data so the build process takes a reasonable amount of time
# resource: https://www.geeksforgeeks.org/machine-learning/python-image-classification-using-keras/

train_sample_size = 10000
validation_sample_size = 2000
epochs = 100 # probably gonna lower it to 50 or 60, but for now let's see how it goes

input_shape = (*IMG_SIZE, 3)   # image_dataset_from_directory always yields channels-last

AUTOTUNE = tf.data.AUTOTUNE

# Rescale pixels to [0, 1]
normalize = lambda x, y: (x / 255.0, y)

# Random flips/rotation/contrast, only applied to training data, to help the model generalize
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomContrast(0.1),
])

# .take() counts batches, not images. The file list was shuffled once with SEED,
# so the first N batches are a random sample. .cache() then freezes that sample
# so every epoch sees the same images (~2 GB of RAM for the training subset;
# lower train_sample_size if that is too much).
train_ds = (train_full_ds
            .take(train_sample_size // batch_size)
            .map(normalize, num_parallel_calls=AUTOTUNE)
            .cache()
            .map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
            .prefetch(AUTOTUNE))

val_ds = (val_full_ds
          .take(validation_sample_size // batch_size)
          .map(normalize, num_parallel_calls=AUTOTUNE)
          .cache()
          .prefetch(AUTOTUNE))

# Full test set, same preprocessing, for model.evaluate() later
test_ds = test_ds.map(normalize, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

# Bumping up the kernel size
model = Sequential()
model.add(Conv2D(32, (3, 3), padding="same", input_shape=input_shape))
model.add(BatchNormalization())  # stabilizes training after each conv layer
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Conv2D(32, (3, 3), padding="same"))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Conv2D(64, (3, 3), padding="same"))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Flatten())
model.add(Dense(64))
model.add(Activation('relu'))
model.add(Dropout(0.5))
model.add(Dense(1))
model.add(Activation('sigmoid'))

model.compile(loss='binary_crossentropy',
              optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),  # Higher initial LR which anneals as val loss plateaus
              metrics=['accuracy'])

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=8,restore_best_weights=True,verbose=1)
live_plot = LiveTrainingPlot()
# Stop early if val_loss stops improving, and restore the best epoch's weights
callbacks = [
    early_stopping, live_plot
]

history = model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=callbacks) # Save the training history

In [ ]:
best_epoch = np.argmin(history.history['val_loss'])

print(f"Best Epoch: {best_epoch + 1}")
print(f"Best Training Loss: {history.history['loss'][best_epoch]}")
print(f"Best Validation Loss: {history.history['val_loss'][best_epoch]}")
print(f"Best Training Accuracy: {history.history['accuracy'][best_epoch]}")
print(f"Best Validation Accuracy: {history.history['val_accuracy'][best_epoch]}")
